# Group 8 — ISIC skin-lesion classification

Coding component: one ResNet-50 classifier over the fixed seven-class label space
`MEL, NV, BCC, AKIEC, BKL, DF, VASC`.

**Data.** ISIC 2018 (HAM10000) supplies the internal lesion-disjoint validation and test splits. The
deduplicated ISIC 2019 images are merged into the training set, which triples the melanoma images and
lowers the nevus share from 67% to 52%; a held-out part of ISIC 2019 stays unseen during training and
is used for external validation. The third dataset listed in the client document (SIIM-ISIC 2020) is
not used: its archive is over 100 GB.

**Training.** Runs on a Google Colab T4 GPU with mixed precision. Three models with different seeds
are trained for exactly 100 epochs each (3 head epochs, 85 fine-tuning epochs at 224 px and 12 at
320 px, progressive resizing) and their softmax outputs are averaged at prediction time (ensemble),
with test-time augmentation and SWA. Every epoch saves its weights and one history row, so a
disconnected session resumes where it stopped.

Every cell runs in order from top to bottom and verifies its own output with assertions.

**Run record** — RUN_RECORD_PLACEHOLDER


## Run plan (Google Colab, free tier T4)

1. Open this notebook in Colab and choose `Runtime → Change runtime type → T4 GPU`.
2. Put your Kaggle API token at hand: Kaggle → `Settings → API Tokens → Create New Token` downloads
   `kaggle.json`. Section 0 asks for it once, then downloads and verifies both datasets (about 6 GB,
   a few minutes) instead of any manual data upload.
3. `Runtime → Run all`. Results are written to `MyDrive/Group_8/outputs`, and about 1 GB of weights and
   one history row are flushed to Drive every few epochs, so a disconnected session resumes where it
   stopped when the training cell is run again.
4. Training is three seeds × 70 epochs, roughly 2.5 hours per seed on a T4.
5. Sections 2.14–2.20 add about 20 minutes for internal test, external validation, TTA, calibration
   and the results archive. The whole notebook takes 6–8 hours on a free T4 session.


## 0. Runtime, paths and reproducibility

The notebook runs in two environments and picks the right one automatically:

| environment | data | artifacts | precision |
|---|---|---|---|
| Google Colab (GPU) | copied from `MyDrive/Group_8/data` to `/content/data` | `MyDrive/Group_8/outputs` | mixed float16 |
| local Anaconda / Jupyter | `data/` next to the notebook | `outputs/` | float32 |

Produces: fixed paths, seeds, GPU check, precision policy and a dataset file count.


In [ ]:
# Section 0 - runtime, paths and seeds. The notebook picks the Colab or the local configuration itself.
import os, sys, json, math, time, hashlib, shutil, subprocess, zipfile, platform, random
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")
from pathlib import Path
import numpy as np
import pandas as pd
import tensorflow as tf

IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = Path("/kaggle/input").is_dir()


def local_project_root(start):
    """Return the project root whether the kernel starts in the root or in code/."""
    for candidate in (start, *start.parents):
        if (candidate / "data").is_dir() or (candidate / "AGENTS.md").is_file():
            return candidate
    raise RuntimeError("open this notebook inside the project folder (data/ not found)")


KAGGLE_2018 = "kmader/skin-cancer-mnist-ham10000"
KAGGLE_2019 = ""          # filled automatically from the Kaggle search API when left empty


def kaggle_token():
    """Return the path of kaggle.json, putting the uploaded token where the CLI expects it."""
    token = Path("/content/kaggle.json")
    if not token.is_file():
        from google.colab import files
        print("upload your kaggle.json (Kaggle -> Settings -> API Tokens -> Create New Token)")
        for name, payload in files.upload().items():
            Path("/content", name).write_bytes(payload)
    assert token.is_file(), "kaggle.json is required to download the datasets"
    os.environ["KAGGLE_CONFIG_DIR"] = "/content"
    return token


def resolve_2019_slug():
    """Search the Kaggle API for the ISIC 2019 classification dataset and take the best match."""
    if KAGGLE_2019:
        return KAGGLE_2019
    result = subprocess.run(["kaggle", "datasets", "list", "-s", "ISIC 2019 skin lesion", "--csv"],
                            capture_output=True, text=True, check=True)
    references = [line.split(",")[0] for line in result.stdout.splitlines()[1:] if line.strip()]
    print("Kaggle candidates for ISIC 2019:", references[:8])
    for reference in references:
        if "isic" in reference.lower() and "2019" in reference.lower():
            return reference
    raise RuntimeError("could not resolve the ISIC 2019 dataset; set KAGGLE_2019 explicitly")


def download_datasets():
    """Download both datasets from Kaggle into /content/data and verify the file counts."""
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kaggle"], check=True)
    kaggle_token()
    targets = {"ISIC 2018": KAGGLE_2018, "ISIC 2019": resolve_2019_slug()}
    for name, reference in targets.items():
        folder = DATA_ROOT / name
        folder.mkdir(parents=True, exist_ok=True)
        if not any(folder.rglob("*.jpg")):
            print(f"downloading {reference} -> {folder} ...")
            subprocess.run(["kaggle", "datasets", "download", "-d", reference, "-p", str(folder), "--unzip"],
                           check=True)
        print(f"{name:10s} {len(list(folder.rglob('*.jpg'))):6d} jpg")


def prepare_bcn(target=None):
    """Locate the optional BCN20000 images (a folder or a zip) in any runtime and extract them."""
    target = Path(target or (DATA_ROOT / "BCN20000"))
    target.mkdir(parents=True, exist_ok=True)
    if any(target.rglob("*.jpg")):
        return target
    archives = []
    for base in (target, DATA_ROOT / "BCN20000"):
        if Path(base).is_dir():
            archives += list(Path(base).rglob("*.zip"))
    if IN_KAGGLE:
        for base in sorted(Path("/kaggle/input").iterdir()):
            archives += list(base.rglob("*.zip"))
    for archive in archives:
        with zipfile.ZipFile(archive) as handle:
            if not any(name.endswith("ISIC_2019_Test_GroundTruth.csv") for name in handle.namelist()):
                continue
            print("extracting", archive.name, "->", target)
            handle.extractall(target)
            break
    return target if any(target.rglob("*.jpg")) else None


def cache_imagenet_weights():
    """Seed the Keras weight cache from an attached dataset so offline runtimes need no download."""
    target = Path.home() / ".keras" / "models"
    target.mkdir(parents=True, exist_ok=True)
    if IN_KAGGLE:
        for source in sorted(Path("/kaggle/input").rglob("*.h5")):
            if "weights_tf_dim_ordering" not in source.name:
                continue
            destination = target / source.name
            if not destination.is_file():
                shutil.copy2(source, destination)
                print("cached pretrained weights:", source.name)


def dataset_root(name, marker, extra=()):
    """Find the folder that holds `marker` for a dataset in any supported runtime."""
    for base in (DATA_ROOT / name, *extra):
        if Path(base).is_dir():
            hits = list(Path(base).rglob(marker))
            if hits:
                return hits[0].parent
    if IN_KAGGLE:
        for base in sorted(Path("/kaggle/input").iterdir()):
            hits = list(base.rglob(marker))
            if hits:
                return hits[0].parent
    return DATA_ROOT / name


if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT_DIR = Path("/content/drive/MyDrive/Group_8/outputs")
    DATA_ROOT = Path("/content/data")
    DATA_ROOT.mkdir(parents=True, exist_ok=True)
    if not any(DATA_ROOT.rglob("*.jpg")):
        download_datasets()
    if not any((DATA_ROOT / "BCN20000").rglob("*.jpg")):
        candidates = list(Path("/content/drive/MyDrive").rglob("ISIC_2019_Test_GroundTruth.csv"))
        if candidates:
            source = candidates[0].parent
            (DATA_ROOT / "BCN20000").mkdir(parents=True, exist_ok=True)
            for path in source.iterdir():
                if path.is_file():
                    shutil.copy2(path, DATA_ROOT / "BCN20000" / path.name)
    BCN_ROOT = prepare_bcn()
    assert BCN_ROOT is not None, "BCN20000 not found; upload it to MyDrive/Group_8/data/BCN20000 or attach the Kaggle dataset"
    HAM_ROOT, ISIC2019_ROOT = DATA_ROOT / "ISIC 2018", DATA_ROOT / "ISIC 2019"
elif IN_KAGGLE:
    DATA_ROOT = Path("/kaggle/working/data")
    DATA_ROOT.mkdir(parents=True, exist_ok=True)
    OUT_DIR = Path("/kaggle/working/outputs")
    HAM_ROOT = dataset_root("ISIC 2018", "HAM10000_metadata.csv")
    ISIC2019_ROOT = dataset_root("ISIC 2019", "ISIC_2019_Training_GroundTruth.csv")
    BCN_ROOT = prepare_bcn()
    assert HAM_ROOT.is_dir(), "attach the HAM10000 dataset to this Kaggle kernel"
    assert ISIC2019_ROOT.is_dir(), "attach the ISIC 2019 dataset to this Kaggle kernel"
else:
    os.chdir(local_project_root(Path.cwd()))
    DATA_ROOT = Path("data")
    OUT_DIR = Path("outputs")
    HAM_ROOT, ISIC2019_ROOT = DATA_ROOT / "ISIC 2018", DATA_ROOT / "ISIC 2019"
    BCN_ROOT = prepare_bcn()

FIGURES = OUT_DIR / "figures"
TABLES = OUT_DIR / "tables"
for folder in (FIGURES, TABLES, OUT_DIR / "model"):
    folder.mkdir(parents=True, exist_ok=True)
cache_imagenet_weights()

CLASS_NAMES = ["MEL", "NV", "BCC", "AKIEC", "BKL", "DF", "VASC"]
SIZE_LOW, SIZE_HIGH = 224, 352          # progressive resizing: most epochs at 224, the tail at 352
BATCH_LOW, BATCH_HIGH = 32, 16
SEEDS = [8, 42, 2026]                    # one model per seed, averaged at prediction time
SEED = 8                                 # fixed data/demo seed: every kernel must see the same split

HAS_GPU = bool(tf.config.list_physical_devices("GPU"))
if HAS_GPU:
    tf.keras.mixed_precision.set_global_policy("mixed_float16")
tf.config.optimizer.set_experimental_options({"disable_meta_optimizer": True})
random.seed(SEED)
np.random.seed(SEED)

print("TensorFlow", tf.__version__, "| Keras", tf.keras.__version__)
print("devices:", tf.config.list_physical_devices())
print("precision policy:", tf.keras.mixed_precision.global_policy().name)
print("data root:", DATA_ROOT.resolve())
print("output root:", OUT_DIR.resolve())
if IN_COLAB:
    assert HAS_GPU, "Colab has no GPU: Runtime -> Change runtime type -> T4 GPU, then rerun"

assert DATA_ROOT.is_dir(), "data folder not found"
# First sanity check: every line must show a number larger than 0.
for name in ("ISIC 2018", "ISIC 2019"):
    print(f"{name:10s} {len(list((DATA_ROOT / name).rglob('*.jpg'))):6d} jpg")


## 2.1 Libraries

Required: list every library used and its version. Produces: printed version table saved to `env.json`.

In [ ]:
# 2.1 - Record every library used and its version (saved to outputs/tables/env.json).
import warnings
warnings.filterwarnings("ignore")
import json, math, platform, time
import tensorflow as tf
import keras
import sklearn
import sklearn.metrics
from sklearn.model_selection import train_test_split
import PIL
import h5py
import matplotlib
import matplotlib.pyplot as plt
from PIL import Image

ENV = {
    "python": platform.python_version(),
    "tensorflow": tf.__version__,
    "keras": keras.__version__,
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "scikit-learn": sklearn.__version__,
    "matplotlib": matplotlib.__version__,
    "pillow": PIL.__version__,
    "h5py": h5py.__version__,
}
env_df = pd.DataFrame({"library": list(ENV), "version": list(ENV.values())})
print(env_df.to_string(index=False))
(TABLES / "env.json").write_text(json.dumps(ENV, indent=2))

# One seed for python, numpy and tensorflow; section 0 already fixed SEED.
tf.keras.utils.set_random_seed(SEED)
print("\nrandom seed:", SEED)


## 2.2 Forward pass and automatic differentiation

Required: explain how the framework tracks the forward pass and recovers gradients. Produces: per-layer gradient norms for one real batch.

**What the tape records.** While the `GradientTape` context is open, TensorFlow stores every operation whose inputs include a trainable variable together with the tensors needed to differentiate it. Closing the context freezes that recording; `tape.gradient()` then replays it in reverse (chain rule) and returns one gradient tensor per trainable variable. The tape keeps references to intermediate activations until it is consumed, which is why training needs more memory than inference.


In [ ]:
# 2.2 - A real image batch, one real forward pass, and the gradients the tape recovers.
def load_raw_batch(folder, n=8):
    """Read JPEGs straight from disk: this is the autodiff demo, full preprocessing comes in 2.4."""
    paths = sorted(p for p in Path(folder).rglob("*.jpg"))[:n]
    batch = [np.asarray(Image.open(p).convert("RGB").resize((SIZE_LOW, SIZE_LOW)), dtype=np.float32) / 255.0
             for p in paths]
    return tf.convert_to_tensor(np.stack(batch)), paths

demo_x, demo_paths = load_raw_batch(HAM_ROOT)
assert len(demo_paths) == 8, "the demo batch needs eight HAM10000 images"
demo_y = tf.one_hot([0, 1, 2, 3, 4, 5, 6, 0], depth=len(CLASS_NAMES))

demo_model = tf.keras.Sequential([
    tf.keras.layers.Input((SIZE_LOW, SIZE_LOW, 3)),
    tf.keras.layers.Conv2D(16, 3, strides=2, activation="relu"),
    tf.keras.layers.Conv2D(32, 3, strides=2, activation="relu"),
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dense(len(CLASS_NAMES)),
], name="tape_demo")
demo_loss = tf.keras.losses.CategoricalCrossentropy(from_logits=True)

# Every operation that depends on a trainable variable is recorded while the context is open.
with tf.GradientTape() as tape:
    logits = demo_model(demo_x, training=True)
    loss = demo_loss(demo_y, logits)

# gradient() replays the recorded graph backwards (chain rule) and returns one tensor per variable.
grads = tape.gradient(loss, demo_model.trainable_variables)
print("demo batch:", tuple(demo_x.shape), "| loss:", round(float(loss), 4))

grad_rows = [{"layer": var.name, "shape": tuple(var.shape),
              "grad_norm": float(tf.norm(g)) if g is not None else 0.0}
             for var, g in zip(demo_model.trainable_variables, grads)]
grad_df = pd.DataFrame(grad_rows)
print(grad_df.to_string(index=False))
grad_df.to_csv(OUT_DIR / "tables" / "gradient_norms_2_2.csv", index=False)


## 2.3 Training and evaluation memory

Required: explain the memory difference between training and evaluation mode. Produces: behaviour evidence and written comparison.

**Why training costs more memory.** In training mode the framework keeps every intermediate activation needed by the backward pass, BatchNormalization uses the statistics of the current batch, and dropout samples a new mask on every call. In evaluation mode no backward pass follows, so activations can be released as soon as they are produced, BatchNormalization reuses the moving statistics learned during training, and dropout is switched off - the output is deterministic and no tape is needed. The printed comparison shows the two regimes: training-mode calls disagree with each other, evaluation-mode calls are identical.


In [ ]:
# 2.3 - Training mode versus evaluation mode: behaviour evidence and cost evidence.
demo_model3 = tf.keras.Sequential([
    tf.keras.layers.Input((SIZE_LOW, SIZE_LOW, 3)),
    tf.keras.layers.Conv2D(16, 3, strides=2, activation="relu"),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dense(len(CLASS_NAMES)),
], name="mode_demo")

train_out_a = demo_model3(demo_x, training=True)
train_out_b = demo_model3(demo_x, training=True)
eval_out_a = demo_model3(demo_x, training=False)
eval_out_b = demo_model3(demo_x, training=False)
print("training=True  -> max |call1 - call2|:", round(float(tf.reduce_max(tf.abs(train_out_a - train_out_b))), 6))
print("training=False -> max |call1 - call2|:", round(float(tf.reduce_max(tf.abs(eval_out_a - eval_out_b))), 6))

@tf.function
def forward_eval(x):
    return demo_model3(x, training=False)

@tf.function
def forward_train_with_tape(x):
    with tf.GradientTape() as tape:
        y = demo_model3(x, training=True)
        loss = tf.reduce_mean(y)
    return tape.gradient(loss, demo_model3.trainable_variables)

def time_forward(fn, repeats=3):
    fn(demo_x)                      # first call traces the tf.function
    start = time.perf_counter()
    for _ in range(repeats):
        fn(demo_x)
    return (time.perf_counter() - start) / repeats

t_eval = time_forward(forward_eval)
t_train = time_forward(forward_train_with_tape)
print(f"eval forward: {t_eval * 1000:.1f} ms | training forward + backward: {t_train * 1000:.1f} ms")
print(f"one training step costs {t_train / t_eval:.1f}x the evaluation pass on this batch")


## Data layer: ISIC 2018 for validation/test, ISIC 2019 merged into training

Rules applied here:
- fixed 7-class label space; HAM10000 lowercase codes are mapped to class names;
- lesion-level, diagnosis-stratified split of ISIC 2018 into train / validation / test;
- SCC dropped from ISIC 2019 (the model has no SCC slot);
- ISIC 2019 images are hash-deduplicated against ISIC 2018 (it largely repeats them);
- the remaining ISIC 2019 images are split 65/35: the larger part joins the training set, the smaller
  part is held out as the external validation set and is never seen during training.

Produces: the merged `train_df`, the internal `val_df` / `test_df`, the external `external_df`, a
split manifest and a data-provenance file.


In [ ]:
# 2.4 - ISIC 2018 (HAM10000): build image path + 7-class label + lesion id.
# HAM10000 stores lowercase codes such as "mel"; map them to the fixed class names.
DX_TO_CLASS = {"mel": "MEL", "nv": "NV", "bcc": "BCC", "akiec": "AKIEC",
               "bkl": "BKL", "df": "DF", "vasc": "VASC"}
EXPECTED_2018_CLASS_COUNTS = {"MEL": 1113, "NV": 6705, "BCC": 514, "AKIEC": 327,
                              "BKL": 1099, "DF": 115, "VASC": 142}

base = HAM_ROOT
meta_path = next(base.rglob("HAM10000_metadata.csv"), None)
assert meta_path is not None, "data/ISIC 2018 must contain HAM10000_metadata.csv"
meta = pd.read_csv(meta_path)
jpgs = {p.stem: p for p in base.rglob("*.jpg")}     # image name -> full path

isic2018 = meta[["image_id", "lesion_id", "dx"]].rename(columns={"image_id": "image"})
isic2018["label"] = isic2018["dx"].map(DX_TO_CLASS)
isic2018["path"] = isic2018["image"].map(lambda name: str(jpgs[name]))
print(isic2018["label"].value_counts().to_string())

# --- verification: the table must describe the complete HAM10000 collection exactly once.
assert len(isic2018) == 10015, f"expected 10015 HAM10000 rows, found {len(isic2018)}"
assert isic2018["image"].is_unique, "duplicate image identifiers"
assert isic2018["label"].notna().all(), "an unknown dx code was mapped to NaN"
assert isic2018["label"].nunique() == len(CLASS_NAMES)
assert isic2018["lesion_id"].nunique() == 7470, "HAM10000 contains 7470 lesions"
assert isic2018.groupby("lesion_id")["label"].nunique().max() == 1, "one lesion has two diagnoses"
assert (isic2018["label"].value_counts().reindex(CLASS_NAMES).tolist()
        == list(EXPECTED_2018_CLASS_COUNTS.values())), "class counts differ from the official release"
missing_files = [p for p in isic2018["path"] if not Path(p).is_file() or Path(p).stat().st_size == 0]
assert not missing_files, f"{len(missing_files)} images are missing, first: {missing_files[:3]}"
assert len(list(base.rglob("*.jpg"))) >= len(isic2018), "more JPEGs than metadata rows: check data/ layout"
print(f"ISIC 2018 verified: {len(isic2018)} images, {isic2018['lesion_id'].nunique()} lesions")


In [ ]:
# Lesion-level split: all images of one lesion land in the same split (no leakage between splits).
# The split is stratified by diagnosis, so every split keeps the class proportions of HAM10000.
lesions = (isic2018[["lesion_id", "label"]].drop_duplicates()
           .sort_values("lesion_id").reset_index(drop=True))
assert len(lesions) == 7470 and lesions["lesion_id"].is_unique

trainval_lesions, test_lesions = train_test_split(
    lesions, test_size=0.15, stratify=lesions["label"], random_state=SEED)
train_lesions, val_lesions = train_test_split(
    trainval_lesions, test_size=0.15 / 0.85, stratify=trainval_lesions["label"], random_state=SEED)

split_by_lesion = {lesion: "train" for lesion in train_lesions["lesion_id"]}
split_by_lesion.update({lesion: "validation" for lesion in val_lesions["lesion_id"]})
split_by_lesion.update({lesion: "test" for lesion in test_lesions["lesion_id"]})
isic2018["split"] = isic2018["lesion_id"].map(split_by_lesion)
print(isic2018["split"].value_counts().to_string())

# --- verification: expected sizes, class coverage and zero leakage between the three splits.
split_frames = {name: isic2018[isic2018["split"] == name] for name in ("train", "validation", "test")}
image_counts = [len(frame) for frame in split_frames.values()]
lesion_counts = [frame["lesion_id"].nunique() for frame in split_frames.values()]
assert image_counts == [7028, 1488, 1499], f"unexpected image split sizes {image_counts}"
assert lesion_counts == [5228, 1121, 1121], f"unexpected lesion split sizes {lesion_counts}"
assert sum(image_counts) == 10015 and sum(lesion_counts) == 7470, "split does not cover the dataset"
for name, frame in split_frames.items():
    assert set(frame["label"]) == set(CLASS_NAMES), f"{name} split misses a diagnosis class"
for first, second in (("train", "validation"), ("train", "test"), ("validation", "test")):
    assert set(split_frames[first]["lesion_id"]).isdisjoint(split_frames[second]["lesion_id"]), \
        f"{first} and {second} share a lesion"
    assert set(split_frames[first]["image"]).isdisjoint(split_frames[second]["image"]), \
        f"{first} and {second} share an image"
print("split sizes:", dict(zip(split_frames, image_counts)))

isic2018.to_csv(OUT_DIR / "tables" / "isic2018_split.csv", index=False)
(OUT_DIR / "tables" / "data_provenance.json").write_text(json.dumps({
    "dataset": "ISIC 2018 Task 3 / HAM10000",
    "external_dataset": "ISIC 2019 training set (8 classes, SCC excluded for the 7-class mapping)",
    "source": "Kaggle archives unpacked manually into data/ (the notebook contains no download code)",
    "license": "CC BY-NC 4.0",
    "seed": SEED,
    "split_unit": "lesion_id",
    "split_protocol": "lesion-level 70/15/15, stratified by diagnosis",
    "split_sizes": dict(zip(split_frames, image_counts)),
}, indent=2))


In [ ]:
# 2.4 - ISIC 2019: eight classes, and SCC has no slot in our seven classes.
# The images are hash-deduplicated against ISIC 2018 first, then split into extra training data and a
# held-out external validation set, both stratified by diagnosis.
ISIC2019_CLASSES = {"MEL": "MEL", "NV": "NV", "BCC": "BCC", "AK": "AKIEC",
                    "BKL": "BKL", "DF": "DF", "VASC": "VASC", "SCC": "SCC"}
EXPECTED_2019_EXTRA = {"MEL": 2210, "NV": 4004, "BCC": 1821, "AKIEC": 474,
                       "BKL": 982, "DF": 81, "VASC": 72}
EXPECTED_2019_EXTERNAL = {"MEL": 1190, "NV": 2156, "BCC": 981, "AKIEC": 255,
                          "BKL": 529, "DF": 43, "VASC": 39}

base = ISIC2019_ROOT
jpgs = {p.stem: p for p in base.rglob("*.jpg")}
truth_path = next(base.rglob("ISIC_2019_Training_GroundTruth.csv"), None)
assert truth_path is not None, "data/ISIC 2019 must contain ISIC_2019_Training_GroundTruth.csv"
truth = pd.read_csv(truth_path)

# --- verification: the ground truth is one-hot and the 7-class mapping is complete.
one_hot = truth[list(ISIC2019_CLASSES)].to_numpy()
assert one_hot.shape == (25331, 8), f"unexpected ISIC 2019 ground-truth shape {one_hot.shape}"
assert (one_hot.sum(axis=1) == 1).all(), "ISIC 2019 ground truth is not one-hot"
truth["label"] = truth[list(ISIC2019_CLASSES)].idxmax(axis=1).map(ISIC2019_CLASSES)
assert truth["label"].notna().all()
scc_count = int((truth["label"] == "SCC").sum())
assert scc_count == 628, f"unexpected SCC count {scc_count}"

isic2019 = truth[truth["label"] != "SCC"][["image", "label"]].copy()
isic2019["path"] = isic2019["image"].map(lambda name: str(jpgs[name]))
assert len(isic2019) == 24703 and isic2019["label"].nunique() == 7
missing_files = [p for p in isic2019["path"] if not Path(p).is_file() or Path(p).stat().st_size == 0]
assert not missing_files, f"{len(missing_files)} ISIC 2019 images are missing, first: {missing_files[:3]}"


def md5(path):
    with open(path, "rb") as handle:
        return hashlib.md5(handle.read()).hexdigest()


isic2018["md5"] = [md5(p) for p in isic2018["path"]]
isic2019["md5"] = [md5(p) for p in isic2019["path"]]

# ISIC 2019 repeats some of its own images, so remove the internal duplicates first: a repeated file
# must never end up both in the training set and in the held-out external set.
internal_duplicates = int(isic2019["md5"].duplicated().sum())
isic2019 = isic2019.drop_duplicates("md5").reset_index(drop=True)
assert len(isic2019) == 24653, f"unexpected size after internal dedup {len(isic2019)}"
assert isic2019["md5"].is_unique

duplicates = isic2019["md5"].isin(isic2018["md5"])
isic2019 = isic2019[~duplicates].reset_index(drop=True)
assert len(isic2019) == 14837, f"unexpected deduplicated size {len(isic2019)}"
assert not isic2019["md5"].isin(isic2018["md5"]).any(), "external set still overlaps ISIC 2018"

extra_rows, external_rows = train_test_split(
    isic2019, test_size=0.35, stratify=isic2019["label"], random_state=SEED)
isic2019["split"] = "extra_train"
isic2019.loc[external_rows.index, "split"] = "external_val"

extra = isic2019[isic2019["split"] == "extra_train"]
external = isic2019[isic2019["split"] == "external_val"]
assert len(extra) == 9644 and len(external) == 5193, (len(extra), len(external))
assert extra["label"].value_counts().reindex(CLASS_NAMES).tolist() == list(EXPECTED_2019_EXTRA.values())
assert external["label"].value_counts().reindex(CLASS_NAMES).tolist() == list(EXPECTED_2019_EXTERNAL.values())
assert set(extra.index).isdisjoint(external.index)
print("ISIC 2019 internal duplicates removed:", internal_duplicates,
      "| images also present in ISIC 2018:", int(duplicates.sum()))
print("ISIC 2019 deduplicated:", len(isic2019))
print("  extra training images:", len(extra), "| held-out external images:", len(external))


In [ ]:
# Merge the extra ISIC 2019 images into the training split. Validation and test stay ISIC 2018 only,
# so the internal metrics remain comparable with the earlier experiments.
train_2018 = isic2018[isic2018["split"] == "train"]
extra_2019 = isic2019[isic2019["split"] == "extra_train"]
COLUMNS = ["image", "path", "label", "md5"]
train_df = pd.concat([train_2018[COLUMNS].assign(source="ISIC2018"),
                      extra_2019[COLUMNS].assign(source="ISIC2019")], ignore_index=True)
val_df = isic2018[isic2018["split"] == "validation"][COLUMNS].assign(source="ISIC2018").reset_index(drop=True)
test_df = isic2018[isic2018["split"] == "test"][COLUMNS].assign(source="ISIC2018").reset_index(drop=True)
external_df = isic2019[isic2019["split"] == "external_val"][COLUMNS].assign(source="ISIC2019").reset_index(drop=True)

# BCN20000 (ISIC 2019 test images with published labels) is optional extra training data: same
# dermoscopy domain and the same label space, with zero md5 overlap against ISIC 2018/2019.
BCN_EXPECTED = {"MEL": 1327, "NV": 2495, "BCC": 975, "AKIEC": 374, "BKL": 660, "DF": 91, "VASC": 104}
bcn_root = BCN_ROOT if BCN_ROOT is not None else None
if bcn_root is not None:
    bcn_gt = pd.read_csv(next(bcn_root.rglob("ISIC_2019_Test_GroundTruth.csv")))
    bcn_columns = [c for c in ["MEL", "NV", "BCC", "AK", "BKL", "DF", "VASC", "SCC", "UNK"] if c in bcn_gt.columns]
    bcn_gt["raw_label"] = bcn_gt[bcn_columns].idxmax(axis=1)
    usable = bcn_gt[bcn_gt["raw_label"].isin(["MEL", "NV", "BCC", "AK", "BKL", "DF", "VASC"])].copy()
    usable["label"] = usable["raw_label"].map({"MEL": "MEL", "NV": "NV", "BCC": "BCC", "AK": "AKIEC",
                                               "BKL": "BKL", "DF": "DF", "VASC": "VASC"})
    bcn_jpgs = {p.stem: p for p in bcn_root.rglob("*.jpg")}
    usable["path"] = usable["image"].map(lambda name: str(bcn_jpgs[name]))
    usable["md5"] = [md5(path) for path in usable["path"]]
    extra_bcn = usable[~usable["md5"].isin(train_df["md5"])][["image", "path", "label", "md5"]].assign(source="BCN20000")
    counts_bcn = extra_bcn["label"].value_counts().reindex(CLASS_NAMES).to_dict()
    assert counts_bcn == BCN_EXPECTED, f"unexpected BCN class counts {counts_bcn}"
    assert not extra_bcn["md5"].isin(external_df["md5"]).any(), "BCN overlaps the external validation set"
    train_df = pd.concat([train_df, extra_bcn], ignore_index=True)
    print("BCN20000 merged into training:", len(extra_bcn), "images")
    print("  dropped as UNK/SCC:", int((bcn_gt["raw_label"].isin(["UNK", "SCC"])).sum()))


EXPECTED_MERGED_TRAIN = {"MEL": 2998, "NV": 8716, "BCC": 2184, "AKIEC": 698,
                         "BKL": 1739, "DF": 163, "VASC": 174}
EXPECTED_MERGED_WITH_BCN = {"MEL": 4325, "NV": 11211, "BCC": 3159, "AKIEC": 1072,
                            "BKL": 2399, "DF": 254, "VASC": 278}
if bcn_root is not None:
    expected_train = EXPECTED_MERGED_WITH_BCN
    expected_total = 22698
else:
    expected_train = EXPECTED_MERGED_TRAIN
    expected_total = 16672
assert len(train_df) == expected_total, f"unexpected merged training size {len(train_df)}"
assert train_df["label"].value_counts().reindex(CLASS_NAMES).tolist() == list(expected_train.values())
assert len(val_df) == 1488 and len(test_df) == 1499 and len(external_df) == 5193
assert not external_df["md5"].isin(train_df["md5"]).any(), "held-out images leak into training"
assert not external_df["md5"].isin(val_df["md5"]).any()
assert not external_df["md5"].isin(test_df["md5"]).any()


manifest = pd.concat([
    train_df.assign(split="train"), val_df.assign(split="validation"),
    test_df.assign(split="test"), external_df.assign(split="external_val")], ignore_index=True)
manifest.to_csv(TABLES / "split_manifest.csv", index=False)
(TABLES / "data_provenance.json").write_text(json.dumps({
    "dataset": "ISIC 2018 Task 3 / HAM10000 (validation and test) + ISIC 2019 training set (training and external validation)",
    "external_dataset": "ISIC 2019, held-out 35% of the hash-deduplicated images",
    "source": "Kaggle archives, copied from Google Drive",
    "license": "CC BY-NC 4.0",
    "seed": SEED,
    "split_unit": "lesion_id for ISIC 2018; image hash for ISIC 2019",
    "split_protocol": "lesion-level stratified 70/15/15 for ISIC 2018; stratified 65/35 for the deduplicated ISIC 2019 images",
    "sizes": {"train": int(len(train_df)), "validation": int(len(val_df)),
              "test": int(len(test_df)), "external_val": int(len(external_df))},
    "known_limitation": "ISIC 2019 has no lesion identifiers, so its images can only be deduplicated by file hash",
}, indent=2))

counts = pd.crosstab(manifest["label"], manifest["split"]).reindex(CLASS_NAMES)[
    ["train", "validation", "test", "external_val"]]
print(counts.to_string())
print("\nmerged training images:", len(train_df), "| nevus share: %.1f%%" % (100 * sum(train_df["label"] == "NV") / len(train_df)))


## 2.4 Preprocessing and label encoding

Required: preprocessing, augmentation and label encoding. Produces: fixed class order, `tf.data` pipelines, one-hot labels, early examples.

**Preprocessing here.** Each JPEG is decoded and resized by a parallel `tf.data` pipeline; the
pipelines keep the pixels at 0-255 and only add the one-hot label, because the ImageNet normalisation
and the training augmentation are layers inside the model (section 2.7). That means training,
validation, test and single-image inference all pass through exactly the same preprocessing code, and
no stale on-disk image cache can ever be reused. The low-resolution pipelines are used by stages A/B,
the high-resolution pipelines (320 px) by stage C and by every evaluation, and the label encoding is
checked against the split tables.


In [ ]:
# 2.4 - Preprocessing and label encoding. Images are decoded and resized by the tf.data pipeline, the
# labels are one-hot encoded in the fixed class order, and the model applies the ImageNet normalisation.
AUTOTUNE = tf.data.AUTOTUNE
CLASS_TO_INDEX = {name: index for index, name in enumerate(CLASS_NAMES)}


def decode_resize(path, size):
    """JPEG bytes -> float32 RGB tensor of shape (size, size, 3), pixels kept at 0-255."""
    image = tf.io.read_file(path)
    image = tf.image.decode_jpeg(image, channels=3)
    image = tf.image.resize(image, (size, size), method="bilinear")
    return tf.cast(image, tf.float32)


def make_pipeline(frame, size, batch_size, training=False):
    """One tf.data pipeline: decode, resize, one-hot encode, (shuffle,) batch, prefetch."""
    paths = tf.constant(frame["path"].tolist())
    labels = tf.constant([CLASS_TO_INDEX[label] for label in frame["label"]], dtype=tf.int32)
    dataset = tf.data.Dataset.from_tensor_slices((paths, labels))
    # Shuffle the file paths before decoding: shuffling after the decode would buffer the whole split as
    # decoded float32 images (several GB) and can exhaust the Colab memory.
    if training:
        dataset = dataset.shuffle(len(frame), seed=SEED, reshuffle_each_iteration=True)
    dataset = dataset.map(lambda path, label: (decode_resize(path, size), tf.one_hot(label, len(CLASS_NAMES))),
                          num_parallel_calls=AUTOTUNE)
    return dataset.batch(batch_size).prefetch(AUTOTUNE)


train_ds_low = make_pipeline(train_df, SIZE_LOW, BATCH_LOW, training=True)
val_ds_low = make_pipeline(val_df, SIZE_LOW, BATCH_LOW)
train_ds_high = make_pipeline(train_df, SIZE_HIGH, BATCH_HIGH, training=True)
val_ds_high = make_pipeline(val_df, SIZE_HIGH, BATCH_HIGH)
test_ds_high = make_pipeline(test_df, SIZE_HIGH, BATCH_HIGH)
external_ds_high = make_pipeline(external_df, SIZE_HIGH, BATCH_HIGH)

# --- verification: batch shapes, pixel range, one-hot rows and their alignment with the label table.
for images, labels in val_ds_low.take(1):
    print("low-resolution batch:", tuple(images.shape), "| pixels:", (float(tf.reduce_min(images)), float(tf.reduce_max(images))))
    assert images.shape[1:] == (SIZE_LOW, SIZE_LOW, 3) and labels.shape[1] == len(CLASS_NAMES)
    assert float(tf.reduce_max(images)) > 1.5, "pixels must stay at 0-255, the model does the normalisation"
for images, labels in test_ds_high.take(1):
    print("high-resolution batch:", tuple(images.shape))
    assert images.shape[1:] == (SIZE_HIGH, SIZE_HIGH, 3)
assert set(val_df["label"]) == set(CLASS_NAMES), "validation split misses a class"
print("train batches per epoch (low/high):",
      math.ceil(len(train_df) / BATCH_LOW), "/", math.ceil(len(train_df) / BATCH_HIGH))


## 2.5–2.6 Augmentation

Required: show two random augmentations of one sample side by side; explain how augmentation increases variance without changing labels. Produces: augmentation figure.

**Augmentation.** Horizontal/vertical flips and small rotations are applied to training images only.
The layer block sits inside the model (section 2.7), so it follows the model's training flag: active
during training, skipped for validation, test and single-image inference. The transforms change the
pixel statistics of an image - so the model sees extra variance - but never change its diagnosis
label. The cell below prints the training batches per epoch and asserts that two consecutive random
draws of the same sample are actually different, which would fail if the augmentation block had
silently become a no-op.


In [ ]:
# 2.5 - Training-only augmentation. The layer block is part of the model (2.7), so it is applied only
# while the model is in training mode and is skipped for validation, test and single-image inference.
AUGMENT = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal_and_vertical", seed=SEED),
    tf.keras.layers.RandomRotation(0.10, seed=SEED),
], name="augment")

# 2.6 - One training sample and two independent random augmentations of the same sample.
first_train = train_df.iloc[0]
sample = tf.convert_to_tensor(
    np.asarray(Image.open(first_train["path"]).convert("RGB").resize((SIZE_LOW, SIZE_LOW)), dtype=np.float32))
batch = sample[None]
panels = [("original", sample),
          ("augmentation 1", AUGMENT(batch, training=True)[0]),
          ("augmentation 2", AUGMENT(batch, training=True)[0])]
for title, panel in panels:
    assert panel.shape == sample.shape, f"{title} changed the tensor shape"
    assert float(tf.reduce_min(panel)) >= 0.0 and float(tf.reduce_max(panel)) <= 255.0, f"{title} left the pixel range"
assert not np.allclose(panels[1][1].numpy(), panels[2][1].numpy()), "augmentation is not random"
assert not np.allclose(panels[0][1].numpy(), panels[1][1].numpy()), "augmentation did not change the image"

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for axis, (title, panel) in zip(axes, panels):
    axis.imshow(np.clip(panel.numpy(), 0, 255).astype(np.uint8))
    axis.set_title(title)
    axis.axis("off")
fig.suptitle("2.5-2.6  same " + first_train["label"] + " sample, two random augmentations (label unchanged)",
             fontsize=11)
fig.tight_layout()
fig.savefig(FIGURES / "augmentation_examples.png", dpi=150, bbox_inches="tight")
plt.show()
print("training batches per epoch:", math.ceil(len(train_df) / BATCH_LOW),
      "| augmentation layers:", [layer.name for layer in AUGMENT.layers])


## 2.7–2.9 Model, summary and feature maps

Required: build the model, print summary, extract feature maps, explain shapes and backpropagation. Produces: summary file, parameter counts, feature-map grid.

**Graph.** The inputs are 0-255 RGB pixels. The graph applies the training augmentation, converts the
pixels to the caffe-style format the ImageNet weights expect (RGB -> BGR, per-channel mean
subtraction), runs the frozen ResNet-50 in inference mode so its BatchNorm statistics never change,
and finally pools, drops out and classifies. Because augmentation and normalisation are layers of the
graph, every later call with `training=False` - validation, test and single-image inference -
automatically reuses the same preprocessing without augmentation.

**Progressive resizing.** The spatial input is declared as `(None, None, 3)`, so the same weights can
be trained at 224 px and later fine-tuned at 320 px without rebuilding the model; only the
`tf.data` pipeline decides the resolution.

**Reading the model.** ResNet-50 first shrinks the spatial size (224 -> 7) while growing the channel
depth (3 -> 2048), so the GAP layer sees one 2048-d vector per image; the dense head then maps it to 7
probabilities. Backpropagation starts from the softmax/cross-entropy gradient and applies the chain
rule layer by layer, updating the head first and, in stages B and C, the residual blocks. Residual
shortcuts let the gradient flow around blocks instead of only through them.


In [ ]:
# 2.7 - ResNet-50 backbone (ImageNet weights) + GAP + dropout + softmax head.
# The spatial size is left dynamic so the same weights train at 224 px and fine-tune at 320 px.
def build_model(seed):
    """Fresh ImageNet backbone and a new seven-class head, seeded for reproducibility."""
    tf.keras.utils.set_random_seed(seed)
    base = tf.keras.applications.ResNet50(include_top=False, weights="imagenet", input_shape=(None, None, 3))
    base.trainable = False      # stage A trains the head only; stages B and C unfreeze conv4/conv5
    inputs = tf.keras.Input((None, None, 3), dtype="float32", name="rgb_image")
    x = AUGMENT(inputs)
    x = tf.keras.layers.Lambda(tf.keras.applications.resnet50.preprocess_input,
                               dtype="float32", name="imagenet_preprocess")(x)
    x = base(x, training=False)      # inference mode: the ImageNet BatchNorm statistics stay fixed
    x = tf.keras.layers.GlobalAveragePooling2D(name="global_average_pool")(x)
    x = tf.keras.layers.Dropout(0.30, name="classifier_dropout")(x)
    outputs = tf.keras.layers.Dense(len(CLASS_NAMES), activation="softmax", dtype="float32",
                                    name="predictions")(x)
    return tf.keras.Model(inputs, outputs, name=f"resnet50_seed{seed}"), base


model, base_model = build_model(SEED)
summary_lines = []
model.summary(print_fn=summary_lines.append)
summary_text = "\n".join(summary_lines)
(TABLES / "model_summary.txt").write_text(summary_text + "\n")
print(summary_text)
trainable_now = sum(int(np.prod(weight.shape)) for weight in model.trainable_weights)
print(f"\ntotal parameters: {model.count_params():,} | trainable in stage A: {trainable_now:,}")

# 2.7 - Verification: the graph, the frozen backbone and the parameter counts must match the plan.
out_224 = model(np.zeros((1, SIZE_LOW, SIZE_LOW, 3), dtype="float32"))
out_320 = model(np.zeros((1, SIZE_HIGH, SIZE_HIGH, 3), dtype="float32"))
assert out_224.shape == (1, len(CLASS_NAMES)) and out_320.shape == (1, len(CLASS_NAMES))
assert np.allclose(out_224.numpy().sum(axis=1), 1.0, atol=1e-4)
assert not base_model.trainable_variables, "the backbone must be frozen before the head stage"
assert trainable_now == 14_343, f"unexpected number of trainable parameters {trainable_now:,}"
assert model.count_params() == 23_602_055, f"unexpected parameter count {model.count_params():,}"

# 2.8 - Feature maps from one hidden layer for one test image.
for images, labels in test_ds_high.take(1):
    test_image, test_label = images[0], int(tf.argmax(labels[0]))
feature_layer = base_model.get_layer("conv4_block6_out")
feature_extractor = tf.keras.Model(inputs=base_model.input, outputs=feature_layer.output)
preprocessed = tf.keras.applications.resnet50.preprocess_input(np.expand_dims(test_image.numpy(), axis=0))
feature_maps = feature_extractor(tf.convert_to_tensor(preprocessed), training=False)[0].numpy()
print(f"feature map shape: {feature_maps.shape} | test image class: {CLASS_NAMES[test_label]}")
assert feature_maps.ndim == 3 and feature_maps.shape[0] == feature_maps.shape[1]

channels = np.linspace(0, feature_maps.shape[-1] - 1, 16).astype(int)
fig, axes = plt.subplots(4, 4, figsize=(10, 10))
for ax, channel in zip(axes.ravel(), channels):
    ax.imshow(feature_maps[:, :, channel], cmap="viridis")
    ax.set_title(f"channel {channel}", fontsize=8)
    ax.axis("off")
fig.suptitle(f"2.8  feature maps at conv4_block6_out ({feature_maps.shape[0]}x{feature_maps.shape[1]}x{feature_maps.shape[2]})",
             fontsize=11)
fig.tight_layout()
fig.savefig(FIGURES / "feature_maps.png", dpi=150, bbox_inches="tight")
plt.show()

# 2.9 - Why the spatial size shrinks while the channel depth grows (report section 2.9).
print("Input %dx%dx3 -> conv4_block6_out %dx%dx%d: spatial size / %d, channel depth x%.0f"
      % (SIZE_HIGH, SIZE_HIGH, feature_maps.shape[0], feature_maps.shape[1], feature_maps.shape[2],
         SIZE_HIGH // feature_maps.shape[0], feature_maps.shape[2] / 3))


## Part 1.1 support: gradient magnitude per layer, VGG vs ResNet

Required by 1.1: a graph of gradient magnitudes per layer for VGG vs ResNet. Produces: the Part 1 figure and its data table.

**Vanishing gradients in a stack.** The chain rule multiplies the Jacobian of every layer, so repeated factors below 1 shrink the gradient as it travels backwards; in a plain VGG stack the earliest convolutions receive much smaller gradients than the last ones. ReLU keeps the derivative at 0 or 1 and therefore does not attenuate active paths the way saturating sigmoid/tanh units do, but it also switches off paths whose pre-activation is negative. Residual shortcuts add an identity term to the Jacobian, so part of the gradient reaches the early layers without being multiplied by every weight matrix. The figure records the L2 norm of the loss gradient for every trainable weight tensor of VGG16 and ResNet-50 on the same real batch.


In [ ]:
# 1.1 - Gradient magnitude per trainable layer for VGG16 versus ResNet-50, same real batch.
def imagenet_model(builder, preprocess_input, name):
    """Backbone with the ImageNet preprocessing it expects, plus a fresh 7-class head."""
    return tf.keras.Sequential([
        tf.keras.layers.Input((SIZE_LOW, SIZE_LOW, 3)),
        tf.keras.layers.Lambda(lambda x: preprocess_input(x * 255.0)),
        builder(include_top=False, weights="imagenet", input_shape=(SIZE_LOW, SIZE_LOW, 3)),
        tf.keras.layers.GlobalAveragePooling2D(),
        tf.keras.layers.Dense(len(CLASS_NAMES)),
    ], name=name)

def per_layer_gradient_table(net, batch):
    """L2 norm of d(loss)/d(weight) for every conv/dense kernel of one forward+backward pass."""
    labels = tf.one_hot([index % len(CLASS_NAMES) for index in range(batch.shape[0])], len(CLASS_NAMES))
    with tf.GradientTape() as tape:
        loss = tf.keras.losses.CategoricalCrossentropy()(labels, net(batch, training=True))
    gradients = tape.gradient(loss, net.trainable_variables)
    rows = []
    for depth, (variable, gradient) in enumerate(zip(net.trainable_variables, gradients)):
        if gradient is None or len(variable.shape) < 2:      # kernels only, skip biases
            continue
        rows.append({"depth_index": depth, "layer": variable.name,
                     "grad_norm": float(tf.norm(gradient))})
    return pd.DataFrame(rows)

vgg_net = imagenet_model(tf.keras.applications.VGG16, tf.keras.applications.vgg16.preprocess_input, "vgg16_gradients")
resnet_net = imagenet_model(tf.keras.applications.ResNet50, tf.keras.applications.resnet50.preprocess_input, "resnet50_gradients")
vgg_table = per_layer_gradient_table(vgg_net, demo_x).assign(model="VGG16")
resnet_table = per_layer_gradient_table(resnet_net, demo_x).assign(model="ResNet-50")
gradient_table = pd.concat([vgg_table, resnet_table], ignore_index=True)
gradient_table["relative_depth"] = gradient_table.groupby("model")["depth_index"].transform(
    lambda column: column / max(column.max(), 1))
gradient_table.to_csv(OUT_DIR / "tables" / "gradient_magnitudes_1_1.csv", index=False)
print(gradient_table.groupby("model")["grad_norm"].agg(["min", "median", "max"]).to_string())

fig, ax = plt.subplots(figsize=(10, 5))
for model_name, color in (("VGG16", "tab:blue"), ("ResNet-50", "tab:orange")):
    part = gradient_table[gradient_table["model"] == model_name].sort_values("relative_depth")
    ax.plot(part["relative_depth"], part["grad_norm"].clip(lower=1e-12),
            color=color, marker="o", markersize=3, linewidth=1.2, label=model_name)
ax.set_yscale("log")
ax.set_xlabel("relative depth (input side -> output side)")
ax.set_ylabel("gradient L2 norm (log scale)")
ax.set_title("1.1  gradient magnitude per layer: VGG16 vs ResNet-50")
ax.legend()
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(OUT_DIR / "figures" / "gradient_magnitudes_vgg_vs_resnet.png", dpi=150, bbox_inches="tight")
plt.show()

del vgg_net, resnet_net      # the gradient tables and figure are enough; free the two extra models


## 2.10–2.11 Training plan and learning-curve evidence

Required: adaptive optimizer, fixed epochs, loss curves with learning rate or gradient norm, overfitting analysis. Produces: training history and curves.

**Plan per seed (70 epochs).** Stage A trains the head for 3 epochs with the frozen ImageNet
backbone (Adam, `1e-3`). Stage B unfreezes `conv4` and `conv5` for 59 epochs at 224 px with a cosine
rate from `3e-5` to `1e-6`. Stage C keeps the same layers trainable and fine-tunes for 8 epochs at
352 px with a small cosine rate from `3e-6` to `1e-7`, which lets the higher resolution sharpen the
lesion boundaries late in training. `BatchNorm` layers stay frozen throughout, so their moving
statistics keep the ImageNet values; the loss is categorical cross-entropy with 0.1 label smoothing,
which reduces over-confidence on ambiguous lesions.

**Three seeds, one ensemble.** The same 100-epoch plan is repeated for three seeds. After each seed
the best-validation-accuracy checkpoint and the average of the last five epochs (SWA) are compared on
the validation split, the better one is kept, and at prediction time the three models are averaged.
`model.fit` performs the optimisation, so Keras handles mixed-precision loss scaling automatically.

**Logging and resume.** Every epoch writes its weights and appends one row (stage, epoch, resolution,
learning rate, training loss/accuracy, validation loss/accuracy/macro F1, seconds) to
`outputs/tables/train_history_seed<seed>.csv` in Drive, so an interrupted Colab session continues
from the last completed epoch. `Adam` keeps first and second moment estimates per parameter and
normalises each step by them; the cosine schedule controls the overall step size.


In [ ]:
# 2.10 - Training plan. Each seed trains one model for exactly 100 epochs:
#   stage A  3 epochs at 224 px, frozen backbone, Adam 1e-3
#   stage B 85 epochs at 224 px, conv4/conv5 trainable, cosine 3e-5 -> 1e-6
#   stage C 8 epochs at 352 px, same layers, cosine 3e-6 -> 1e-7
# BatchNorm stays frozen, the loss uses 0.1 label smoothing, and the last five epochs are averaged as
# a SWA candidate. model.fit is used so Keras handles the mixed-precision loss scaling automatically.
MODEL_ROOT = OUT_DIR / "model"
STAGE_EPOCHS = {"A": 3, "B": 59, "C": 8}
TOTAL_EPOCHS = sum(STAGE_EPOCHS.values())
STAGE_SIZE = {"A": SIZE_LOW, "B": SIZE_LOW, "C": SIZE_HIGH}
HEAD_LR = 1e-3
STAGE_LR = {"B": (3e-5, 1e-6), "C": (3e-6, 1e-7)}
LABEL_SMOOTHING = 0.1
SWA_LAST = 5
HISTORY_COLUMNS = ["stage", "epoch", "size", "lr", "train_loss", "train_acc",
                   "val_loss", "val_acc", "val_macro_f1", "seconds"]
sample_loss_fn = tf.keras.losses.CategoricalCrossentropy(label_smoothing=LABEL_SMOOTHING, reduction="none")


# 2.10 - Experiment configuration next to the plan (kept in one file for the report).
config = {
    "architecture": "ResNet-50, ImageNet weights, GAP + dropout 0.30 + softmax, dynamic spatial input",
    "datasets": {"train": "ISIC 2018 train + 65% of deduplicated ISIC 2019", "validation": "ISIC 2018 validation",
                 "test": "ISIC 2018 test", "external_val": "held-out 35% of deduplicated ISIC 2019"},
    "seeds": SEEDS, "epochs_per_seed": TOTAL_EPOCHS, "stages": STAGE_EPOCHS,
    "sizes": {"low": SIZE_LOW, "high": SIZE_HIGH}, "batch_sizes": {"low": BATCH_LOW, "high": BATCH_HIGH},
    "head_lr": HEAD_LR, "stage_lr": {stage: list(values) for stage, values in STAGE_LR.items()},
    "optimizer": "Adam", "loss": "categorical cross-entropy with 0.1 label smoothing",
    "fine_tune_scope": "conv4/conv5 trainable, BatchNorm frozen",
    "augmentation": ["RandomFlip horizontal+vertical", "RandomRotation 0.10"],
    "selection": "per seed: better of best-validation checkpoint and SWA on validation accuracy; ensemble averages the three models",
    "predictions": "test-time augmentation over four flip views, averaged over the three seeds",
    "calibration": "single temperature fitted on the validation split",
}
(TABLES / "experiment_config.json").write_text(json.dumps(config, indent=2))
print(json.dumps(config, indent=2)[:1200])

CHECKPOINT_EVERY = 5     # write last.weights.h5 to Drive every few epochs; an interrupted run replays at most 4


def save_checkpoint(model, path):
    """Save weights without the Adam state: Keras stores the slot variables too, which triples the
    file size, and a resumed run simply replays the few epochs since the checkpoint."""
    model.save_weights(path)
    with h5py.File(path, "a") as handle:
        if "optimizer" in handle:
            del handle["optimizer"]


def evaluate(model, dataset):
    """Unweighted loss, accuracy and macro F1 over a tf.data dataset (validation and test only)."""
    total_loss, seen, true_labels, predictions = 0.0, 0, [], []
    for images, labels in dataset:
        probabilities = model(images, training=False)
        total_loss += float(tf.reduce_sum(sample_loss_fn(labels, probabilities)))
        true_labels.append(np.argmax(labels.numpy(), axis=1))
        predictions.append(np.argmax(probabilities.numpy(), axis=1))
        seen += int(images.shape[0])
    true_labels, predictions = np.concatenate(true_labels), np.concatenate(predictions)
    return (total_loss / seen, float(np.mean(true_labels == predictions)),
            float(sklearn.metrics.f1_score(true_labels, predictions, average="macro")))


def learning_rate_for(stage, epoch_in_stage):
    """Stage A is constant; stages B and C follow an epoch-level cosine decay."""
    if stage == "A":
        return HEAD_LR
    start, end = STAGE_LR[stage]
    position = epoch_in_stage / max(1, STAGE_EPOCHS[stage] - 1)
    return float(end + 0.5 * (start - end) * (1 + math.cos(math.pi * position)))


class TrainingCallback(tf.keras.callbacks.Callback):
    """Sets the epoch learning rate, logs one history row and keeps last/best/SWA checkpoints."""

    def __init__(self, seed, stage, size, first_epoch, seed_dir, history_path, val_dataset):
        super().__init__()
        self.seed, self.stage, self.size, self.first_epoch = seed, stage, size, first_epoch
        self.seed_dir = seed_dir
        self.history_path = history_path
        self.val_dataset = val_dataset
        self.last_weights = seed_dir / "last.weights.h5"
        self.best_weights = seed_dir / "best.weights.h5"
        self.best_json = seed_dir / "best.json"
        self.best = json.loads(self.best_json.read_text()) if self.best_json.is_file() else {"val_acc": -1.0}
        self.swa_weights, self.swa_count = None, 0
        self.started = 0.0

    def on_epoch_begin(self, epoch, logs=None):
        self.started = time.perf_counter()
        self.model.optimizer.learning_rate.assign(learning_rate_for(self.stage, epoch))

    def on_epoch_end(self, epoch, logs=None):
        current_epoch = self.first_epoch + epoch
        val_loss, val_acc, val_f1 = evaluate(self.model, self.val_dataset)
        row = {"stage": self.stage, "epoch": current_epoch, "size": self.size,
               "lr": learning_rate_for(self.stage, epoch), "train_loss": float(logs["loss"]),
               "train_acc": float(logs["accuracy"]), "val_loss": val_loss, "val_acc": val_acc,
               "val_macro_f1": val_f1, "seconds": time.perf_counter() - self.started}
        if current_epoch % CHECKPOINT_EVERY == 0 or current_epoch == TOTAL_EPOCHS:
            save_checkpoint(self.model, self.last_weights)
            (self.seed_dir / "last.json").write_text(json.dumps(
                {"seed": self.seed, "stage": self.stage, "epoch": current_epoch}))
        if val_acc > self.best["val_acc"]:
            self.best = {"val_acc": val_acc, "epoch": current_epoch, "stage": self.stage}
            save_checkpoint(self.model, self.best_weights)
            self.best_json.write_text(json.dumps(self.best))

        history = (pd.read_csv(self.history_path) if Path(self.history_path).is_file()
                   else pd.DataFrame(columns=HISTORY_COLUMNS))
        history = pd.concat([history, pd.DataFrame([row])], ignore_index=True)
        history.to_csv(self.history_path, index=False)
        if self.stage == "C" and epoch >= STAGE_EPOCHS["C"] - SWA_LAST:
            weights = self.model.get_weights()
            if self.swa_weights is None:
                self.swa_weights = [w.astype(np.float64) for w in weights]
            else:
                for index, weight in enumerate(weights):
                    self.swa_weights[index] += weight.astype(np.float64)
            self.swa_count += 1
        print(f"  seed {self.seed} | {self.stage} epoch {current_epoch:3d}/{TOTAL_EPOCHS} "
              f"@ {self.size}px | train loss {row['train_loss']:.4f} acc {row['train_acc']:.3f} | "
              f"val loss {val_loss:.4f} acc {val_acc:.4f} f1 {val_f1:.4f} | "
              f"lr {row['lr']:.1e} | {row['seconds']:.0f}s", flush=True)


def fit_stage(model, base_model, seed, stage, seed_dir, history_path, history):
    """Run (or resume) one stage of the 100-epoch plan."""
    done = int((history["stage"] == stage).sum()) if len(history) else 0
    if done >= STAGE_EPOCHS[stage]:
        return done
    if stage == "A":
        base_model.trainable = False
        for layer in base_model.layers:
            layer.trainable = False
    else:
        base_model.trainable = True
        for layer in base_model.layers:
            conv_block = layer.name.startswith(("conv4", "conv5"))
            layer.trainable = conv_block and not isinstance(layer, tf.keras.layers.BatchNormalization)
    size = STAGE_SIZE[stage]
    train_ds = train_ds_low if size == SIZE_LOW else train_ds_high
    val_ds = val_ds_low if size == SIZE_LOW else val_ds_high
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate_for(stage, done)),
                  loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=LABEL_SMOOTHING),
                  metrics=["accuracy"], jit_compile=False)
    first_epoch = sum(STAGE_EPOCHS[s] for s in ("A", "B", "C") if s < stage) + 1
    callback = TrainingCallback(seed, stage, size, first_epoch, seed_dir, history_path, val_ds)
    print(f"\nseed {seed} | stage {stage}: epochs {done + 1}-{STAGE_EPOCHS[stage]} @ {size}px, "
          f"trainable weights {sum(int(np.prod(w.shape)) for w in model.trainable_weights):,}")
    model.fit(train_ds, validation_data=val_ds, epochs=STAGE_EPOCHS[stage], initial_epoch=done,
              callbacks=[callback], verbose=0)
    if stage == "C" and callback.swa_weights is not None:
        averaged = [weight / callback.swa_count for weight in callback.swa_weights]
        current = model.get_weights()
        model.set_weights(averaged)
        save_checkpoint(model, seed_dir / "swa.weights.h5")
        model.set_weights(current)
        print(f"seed {seed}: SWA average of the last {callback.swa_count} epochs saved")
    return STAGE_EPOCHS[stage]


for seed in SEEDS:
    seed_dir = MODEL_ROOT / f"seed{seed}"
    seed_dir.mkdir(parents=True, exist_ok=True)
    history_path = TABLES / f"train_history_seed{seed}.csv"
    model, base_model = build_model(seed)
    history = pd.read_csv(history_path) if history_path.is_file() else pd.DataFrame(columns=HISTORY_COLUMNS)
    print(f"\n=== seed {seed}: {len(history)} epochs already in history ===")
    if len(history) and (seed_dir / "last.weights.h5").is_file():
        marker = json.loads((seed_dir / "last.json").read_text()) if (seed_dir / "last.json").is_file() else {}
        last_epoch = int(marker.get("epoch", 0))
        history = history[history["epoch"] <= last_epoch].reset_index(drop=True)
        history.to_csv(history_path, index=False)
        model.load_weights(seed_dir / "last.weights.h5")
        print(f"resumed weights from {seed_dir / 'last.weights.h5'} (epoch {last_epoch}); "
              f"history truncated to {len(history)} epochs")
    for stage in ("A", "B", "C"):
        history = (pd.read_csv(history_path) if history_path.is_file()
                   else pd.DataFrame(columns=HISTORY_COLUMNS))
        fit_stage(model, base_model, seed, stage, seed_dir, history_path, history)
    history = pd.read_csv(history_path)
    assert len(history) == TOTAL_EPOCHS, f"seed {seed}: {len(history)} epochs recorded, expected {TOTAL_EPOCHS}"
    assert history["epoch"].tolist() == list(range(1, TOTAL_EPOCHS + 1)), "epoch numbering must be continuous"
    assert np.isfinite(history[["train_loss", "val_loss", "train_acc", "val_acc"]].to_numpy()).all()

    # Compare the best-validation checkpoint with the SWA average on the validation split.
    model.load_weights(seed_dir / "best.weights.h5")
    best_eval = evaluate(model, val_ds_high)
    selection = {"seed": seed, "best_val_acc": best_eval[1], "best_epoch": json.loads((seed_dir / "best.json").read_text())["epoch"]}
    swa_path = seed_dir / "swa.weights.h5"
    if swa_path.is_file():
        model.load_weights(swa_path)
        swa_eval = evaluate(model, val_ds_high)
        selection.update(swa_val_acc=swa_eval[1])
        if swa_eval[1] >= best_eval[1]:
            selection["selected"] = "swa"
        else:
            selection["selected"] = "best"
    else:
        selection["selected"] = "best"
    model.load_weights(seed_dir / ("swa.weights.h5" if selection["selected"] == "swa" else "best.weights.h5"))
    save_checkpoint(model, seed_dir / "selected.weights.h5")
    (seed_dir / "selection.json").write_text(json.dumps(selection, indent=2))
    print(f"seed {seed} selected checkpoint: {selection}")


In [ ]:
# Training summary: every seed must have exactly 100 epochs recorded, then the stage means are printed.
# (The SWA average is saved inside fit_stage when stage C ends.)
for seed in SEEDS:
    seed_dir = MODEL_ROOT / f"seed{seed}"
    history_path = TABLES / f"train_history_seed{seed}.csv"
    history = pd.read_csv(history_path)
    assert len(history) == TOTAL_EPOCHS
print("training complete for all seeds:", SEEDS)
print(history.groupby("stage")[["train_loss", "val_loss", "val_acc", "val_macro_f1", "seconds"]].mean().round(4).to_string())


## 2.12–2.13 Learning curves and overfitting analysis

The loss curves are drawn from the per-seed history files, with the learning-rate schedule on the
second axis and vertical lines for the stage boundaries. The second figure plots validation accuracy
and macro F1 per epoch, because accuracy alone hides class-wise regressions on this imbalanced split.
The analysis cell derives every number quoted in section 2.13 from the same tables and writes it to
`outputs/tables/curve_analysis.txt`, so the prose and the recorded evidence cannot drift apart after
a rerun.


In [ ]:
# 2.12 - Per-seed training curves: loss with the learning-rate schedule on a second axis.
fig, axes = plt.subplots(len(SEEDS), 1, figsize=(11, 3.6 * len(SEEDS)), squeeze=False)
for row, seed in enumerate(SEEDS):
    history = pd.read_csv(TABLES / f"train_history_seed{seed}.csv")
    axis = axes[row, 0]
    axis.plot(history["epoch"], history["train_loss"], color="tab:blue", linewidth=1, marker="o", markersize=2, label="training loss")
    axis.plot(history["epoch"], history["val_loss"], color="tab:red", linewidth=1, marker="o", markersize=2, label="validation loss")
    stage_a_end = STAGE_EPOCHS["A"]
    stage_b_end = stage_a_end + STAGE_EPOCHS["B"]
    axis.axvline(stage_a_end + 0.5, color="grey", linestyle="--", alpha=0.7)
    axis.axvline(stage_b_end + 0.5, color="grey", linestyle="--", alpha=0.7)
    best_row = history.loc[history["val_acc"].idxmax()]
    axis.axvline(best_row["epoch"], color="tab:red", linestyle=":", alpha=0.8,
                 label=f"best epoch {int(best_row['epoch'])}")
    axis.set_ylabel("cross-entropy")
    axis.set_title(f"seed {seed}: training / validation loss (stage A | B | C at 224/224/320 px)")
    axis.grid(alpha=0.3)
    rate_axis = axis.twinx()
    rate_axis.plot(history["epoch"], history["lr"], color="tab:green", linestyle="--", linewidth=1, label="learning rate")
    rate_axis.set_yscale("log")
    rate_axis.set_ylabel("learning rate")
    handles = axis.get_lines() + rate_axis.get_lines()
    axis.legend(handles, [handle.get_label() for handle in handles], loc="upper right", fontsize=8)
    if row == len(SEEDS) - 1:
        axis.set_xlabel("epoch")
fig.tight_layout()
fig.savefig(FIGURES / "training_curves.png", dpi=150, bbox_inches="tight")
plt.show()

fig, axes = plt.subplots(1, len(SEEDS), figsize=(5 * len(SEEDS), 3.6), squeeze=False)
for column, seed in enumerate(SEEDS):
    history = pd.read_csv(TABLES / f"train_history_seed{seed}.csv")
    axis = axes[0, column]
    axis.plot(history["epoch"], history["val_acc"], color="tab:blue", linewidth=1, marker="o", markersize=2, label="val accuracy")
    axis.plot(history["epoch"], history["val_macro_f1"], color="tab:purple", linewidth=1, marker="o", markersize=2, label="val macro F1")
    axis.set_title(f"seed {seed}", fontsize=10)
    axis.set_xlabel("epoch")
    axis.grid(alpha=0.3)
    axis.legend(fontsize=8)
fig.tight_layout()
fig.savefig(FIGURES / "validation_metrics.png", dpi=150, bbox_inches="tight")
plt.show()
print("stage summary per seed:")
for seed in SEEDS:
    history = pd.read_csv(TABLES / f"train_history_seed{seed}.csv")
    best = history.loc[history["val_acc"].idxmax()]
    print(f"  seed {seed}: best val acc {best['val_acc']:.4f} at epoch {int(best['epoch'])} "
          f"({int(best['size'])}px, val loss {best['val_loss']:.4f}, macro F1 {best['val_macro_f1']:.4f})")


In [ ]:
# 2.13 - Overfitting evidence derived from the recorded histories (written to a text file).
lines = []
for seed in SEEDS:
    history = pd.read_csv(TABLES / f"train_history_seed{seed}.csv")
    best_row = history.loc[history["val_acc"].idxmax()]
    last_row = history.iloc[-1]
    tail = history.tail(10)
    lowest_loss = history.loc[history["val_loss"].idxmin()]
    best_f1 = history.loc[history["val_macro_f1"].idxmax()]
    warning_windows = []
    for index in range(2, len(history)):
        window = history.iloc[index - 2:index + 1]
        if (window["stage"].nunique() == 1 and (np.diff(window["val_loss"]) > 0).all()
                and (np.diff(window["train_loss"]) < 0).all()):
            warning_windows.append((int(window.iloc[0]["epoch"]), int(window.iloc[-1]["epoch"])))
    lines += [
        f"seed {seed}: {len(history)} epochs, {history['seconds'].sum() / 60:.1f} min of training.",
        f"  best validation accuracy {best_row['val_acc']:.4f} at epoch {int(best_row['epoch'])} "
        f"({int(best_row['size'])} px, validation loss {best_row['val_loss']:.4f}, macro F1 {best_row['val_macro_f1']:.4f}).",
        f"  lowest validation loss {lowest_loss['val_loss']:.4f} at epoch {int(lowest_loss['epoch'])}; "
        f"highest macro F1 {best_f1['val_macro_f1']:.4f} at epoch {int(best_f1['epoch'])}.",
        f"  final epoch: train loss {last_row['train_loss']:.4f} (accuracy {last_row['train_acc']:.3f}), "
        f"validation loss {last_row['val_loss']:.4f} (accuracy {last_row['val_acc']:.3f}); "
        f"gap {last_row['val_loss'] - last_row['train_loss']:+.4f} nat.",
        f"  last ten epochs: validation accuracy range {100 * (tail['val_acc'].max() - tail['val_acc'].min()):.2f} "
        f"percentage points, validation loss range {tail['val_loss'].max() - tail['val_loss'].min():.4f}.",
        "  first window with validation loss rising twice while training loss falls: "
        f"{warning_windows[0] if warning_windows else 'none'} (descriptive, not proof of an onset).",
    ]
analysis = "\n".join(lines)
print(analysis)
(TABLES / "curve_analysis.txt").write_text(analysis + "\n")


## 2.13 Learning-curve analysis

RESULT_PLACEHOLDER_2_13


## 2.14–2.17 Internal test evaluation

Required: test accuracy and loss, normalised confusion matrix, top-3 confident errors, error analysis. Produces: evaluation figures and tables.

**Internal test.** The three selected models are applied to the held-out lesion-disjoint test split
with four-view test-time augmentation, and their softmax outputs are averaged. One ensemble pass
supplies every number in this section: loss, accuracy, balanced accuracy, macro F1, the row-normalised
7x7 confusion matrix and the three most confident mistakes. Per-class precision/recall and the most
frequent confusion pairs are printed and saved, and they are the evidence for the error analysis in
section 2.17.

RESULT_PLACEHOLDER_2_14


In [ ]:
# 2.14 - Ensemble prediction with test-time augmentation. Every selected model is applied to four
# flipped views of each image and the models' softmax outputs are averaged.
def tta_probabilities(model, dataset):
    """Averaged softmax over four flips at the training size and at a slightly smaller size."""
    probabilities, true_labels = [], []
    for images, labels in dataset:
        views = []
        for size in (SIZE_HIGH, SIZE_HIGH - 32):
            base = images if size == SIZE_HIGH else tf.image.resize(images, (size, size))
            views += [base, tf.image.flip_left_right(base), tf.image.flip_up_down(base),
                      tf.image.flip_left_right(tf.image.flip_up_down(base))]
        probabilities.append(np.mean([model(view, training=False).numpy() for view in views], axis=0))
        true_labels.append(labels.numpy().argmax(axis=1))
    return np.concatenate(probabilities), np.concatenate(true_labels)


_SELECTED_MODELS = {}


def selected_model(seed):
    """Build the seed's model once and load its selected checkpoint (cached for later passes)."""
    if seed not in _SELECTED_MODELS:
        model, _ = build_model(seed)
        model.load_weights(MODEL_ROOT / f"seed{seed}" / "selected.weights.h5")
        _SELECTED_MODELS[seed] = model
    return _SELECTED_MODELS[seed]


def ensemble_probabilities(dataset):
    """Average the TTA predictions of the selected checkpoint of every seed."""
    seed_probabilities = []
    for seed in SEEDS:
        probabilities, true_labels = tta_probabilities(selected_model(seed), dataset)
        seed_probabilities.append(probabilities)
        print(f"seed {seed}: ensemble member ready")
    return np.mean(seed_probabilities, axis=0), true_labels


In [ ]:
# 2.15 - Internal test evaluation from one ensemble pass.
test_probs, test_labels = ensemble_probabilities(test_ds_high)
test_rows = test_df.reset_index(drop=True)
assert test_probs.shape == (len(test_rows), len(CLASS_NAMES))
np.testing.assert_allclose(test_probs.sum(axis=1), 1.0, atol=1e-5, err_msg="probabilities do not sum to one")
assert np.array_equal(test_labels, np.array([CLASS_TO_INDEX[label] for label in test_rows["label"]]))

test_pred = test_probs.argmax(axis=1)
test_confidence = test_probs.max(axis=1)
test_loss = float(np.mean(sample_loss_fn(tf.one_hot(test_labels, len(CLASS_NAMES)),
                                         tf.constant(test_probs, dtype=tf.float32)).numpy()))
test_accuracy = float(np.mean(test_pred == test_labels))
test_macro_f1 = float(sklearn.metrics.f1_score(test_labels, test_pred, average="macro"))
test_balanced_accuracy = float(sklearn.metrics.balanced_accuracy_score(test_labels, test_pred))
print(f"internal test ({len(test_rows)} images, {len(SEEDS)}-seed ensemble with TTA): loss {test_loss:.4f} | "
      f"accuracy {test_accuracy:.4f} | balanced accuracy {test_balanced_accuracy:.4f} | macro F1 {test_macro_f1:.4f}")

classification_report = sklearn.metrics.classification_report(
    test_labels, test_pred, target_names=CLASS_NAMES, digits=4, zero_division=0)
(TABLES / "test_classification_report.txt").write_text(classification_report + "\n")
print(classification_report)

# 2.16 - Row-normalised confusion matrix (each row sums to 1).
confusion = sklearn.metrics.confusion_matrix(test_labels, test_pred, labels=np.arange(len(CLASS_NAMES)))
confusion_normalised = confusion / confusion.sum(axis=1, keepdims=True)
np.testing.assert_allclose(confusion_normalised.sum(axis=1), 1.0)
fig, ax = plt.subplots(figsize=(8, 7))
image_handle = ax.imshow(confusion_normalised, cmap="Blues", vmin=0, vmax=1)
for row in range(len(CLASS_NAMES)):
    for column in range(len(CLASS_NAMES)):
        value = confusion_normalised[row, column]
        ax.text(column, row, f"{value:.2f}", ha="center", va="center", fontsize=8,
                color="white" if value > 0.5 else "black")
ax.set_xticks(range(len(CLASS_NAMES)))
ax.set_yticks(range(len(CLASS_NAMES)))
ax.set_xticklabels(CLASS_NAMES, rotation=45)
ax.set_yticklabels(CLASS_NAMES)
ax.set_xlabel("predicted class")
ax.set_ylabel("true class")
ax.set_title("2.16  row-normalised confusion matrix (internal test, ensemble)")
fig.colorbar(image_handle, ax=ax, fraction=0.046)
fig.tight_layout()
fig.savefig(FIGURES / "confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

# 2.17 - The three most confident wrong predictions, chosen by predicted-class probability.
wrong_indices = np.where(test_pred != test_labels)[0]
assert len(wrong_indices) > 0, "no wrong test predictions: the error analysis would be meaningless"
confident_wrong = wrong_indices[np.argsort(test_confidence[wrong_indices])[::-1][:3]]
fig, axes = plt.subplots(1, len(confident_wrong), figsize=(4.5 * len(confident_wrong), 4.5), squeeze=False)
for axis, index in zip(axes.flat, confident_wrong):
    axis.imshow(Image.open(test_rows.loc[index, "path"]).convert("RGB"))
    axis.set_title(f"true: {CLASS_NAMES[test_labels[index]]}\n"
                   f"pred: {CLASS_NAMES[test_pred[index]]} ({test_confidence[index]:.3f})", fontsize=9)
    axis.axis("off")
fig.suptitle("2.17  top-3 most confident wrong predictions (internal test, ensemble)", fontsize=11)
fig.tight_layout()
fig.savefig(FIGURES / "top_confident_errors.png", dpi=150, bbox_inches="tight")
plt.show()
print("most confident errors:", [
    (CLASS_NAMES[test_labels[index]], CLASS_NAMES[test_pred[index]],
     round(float(test_confidence[index]), 3)) for index in confident_wrong])

confusion_pairs = [{"true": CLASS_NAMES[true_index], "predicted": CLASS_NAMES[predicted_index],
                    "count": int(confusion[true_index, predicted_index]),
                    "row_fraction": float(confusion_normalised[true_index, predicted_index])}
                   for true_index in range(len(CLASS_NAMES)) for predicted_index in range(len(CLASS_NAMES))
                   if true_index != predicted_index and confusion[true_index, predicted_index] > 0]
confusion_pairs = pd.DataFrame(confusion_pairs).sort_values("count", ascending=False)
confusion_pairs.to_csv(TABLES / "test_confusion_pairs.csv", index=False)
print(confusion_pairs.head(6).to_string(index=False))

# Full prediction table for the report appendix.
predictions = test_rows[["image", "lesion_id", "path", "label"]].copy() if "lesion_id" in test_rows else test_rows[["image", "path", "label"]].copy()
predictions["true_class"] = [CLASS_NAMES[index] for index in test_labels]
predictions["predicted_class"] = [CLASS_NAMES[index] for index in test_pred]
predictions["confidence"] = test_confidence
for index, name in enumerate(CLASS_NAMES):
    predictions["p_" + name] = test_probs[:, index]
predictions.to_csv(TABLES / "test_predictions.csv", index=False)

np.savez_compressed(TABLES / "test_predictions.npz", probs=test_probs, labels=test_labels)
test_metrics = {"test_loss": test_loss, "test_accuracy": test_accuracy,
                "balanced_accuracy": test_balanced_accuracy, "macro_f1": test_macro_f1,
                "test_images": len(test_rows), "seeds": SEEDS}
(TABLES / "test_metrics.json").write_text(json.dumps(test_metrics, indent=2))


## 2.18–2.20 External dataset, unseen image and calibration

Required: external evaluation, unseen-image probabilities, softmax discussion, calibration. Produces: external table, bar chart, reliability diagram and ECE.

**Softmax and calibration.** The dense head produces logits $z$; softmax maps them to
$p_k = \exp(z_k - m) / \sum_j \exp(z_j - m)$ with $m = \max_j z_j$, so the seven outputs are positive,
sum to one and cannot overflow. A classifier is calibrated when predictions made at 80% confidence are
correct about 80% of the time. The reliability diagram bins test predictions by confidence and
compares mean confidence with empirical accuracy; the expected calibration error (ECE) is the
count-weighted average gap between the two. A single temperature is fitted on the validation split
(never on test) and the ECE is reported before and after that scaling.

**External validation** uses 5,193 held-out ISIC 2019 images that were not part of training: they test
whether the model transfers to a different collection with a different class prior.

RESULT_PLACEHOLDER_2_18


In [ ]:
# 2.18 - External validation on the held-out ISIC 2019 images (never seen during training).
external_probs, external_labels = ensemble_probabilities(external_ds_high)
assert len(external_labels) == len(external_df)
np.testing.assert_allclose(external_probs.sum(axis=1), 1.0, atol=1e-5)
external_pred = external_probs.argmax(axis=1)
external_confidence = external_probs.max(axis=1)
external_loss = float(np.mean(sample_loss_fn(tf.one_hot(external_labels, len(CLASS_NAMES)),
                                            tf.constant(external_probs, dtype=tf.float32)).numpy()))
external_accuracy = float(np.mean(external_pred == external_labels))
external_balanced_accuracy = float(sklearn.metrics.balanced_accuracy_score(external_labels, external_pred))
external_macro_f1 = float(sklearn.metrics.f1_score(external_labels, external_pred, average="macro"))
print(f"ISIC 2019 external ({len(external_labels)} images, ensemble with TTA): loss {external_loss:.4f} | "
      f"accuracy {external_accuracy:.4f} | balanced accuracy {external_balanced_accuracy:.4f} | "
      f"macro F1 {external_macro_f1:.4f}")

external_table = pd.DataFrame({
    "class": CLASS_NAMES,
    "support": [int((external_labels == index).sum()) for index in range(len(CLASS_NAMES))],
    "predicted": [int((external_pred == index).sum()) for index in range(len(CLASS_NAMES))],
    "recall": [float(((external_pred == index) & (external_labels == index)).sum()
                     / max((external_labels == index).sum(), 1)) for index in range(len(CLASS_NAMES))],
})
assert external_table["support"].sum() == len(external_labels)
external_table.to_csv(TABLES / "external_isic2019_per_class.csv", index=False)
print(external_table.to_string(index=False))


# 2.19 - One reusable single-image function plus one fixed unseen ISIC 2019 image.
def predict_image(image_path):
    """Return the seven class probabilities for one JPEG, exactly as the evaluation computes them."""
    image = decode_resize(image_path, SIZE_HIGH)
    views = []
    for size in (SIZE_HIGH, SIZE_HIGH - 32):
        base = image if size == SIZE_HIGH else tf.image.resize(image, (size, size))
        views += [base, tf.image.flip_left_right(base), tf.image.flip_up_down(base),
                  tf.image.flip_left_right(tf.image.flip_up_down(base))]
    probabilities = []
    for seed in SEEDS:
        model = selected_model(seed)
        probabilities.append(np.mean([model(view[None], training=False).numpy()[0] for view in views], axis=0))
    return np.mean(probabilities, axis=0)


reference_row = test_df.reset_index(drop=True).loc[0]
np.testing.assert_allclose(predict_image(reference_row["path"]), test_probs[0], rtol=5e-3, atol=2e-3)
print("single-image inference reproduces the batched ensemble prediction")

unseen_row = external_df.iloc[0]
unseen_probs = predict_image(unseen_row["path"])
unseen_image = np.asarray(Image.open(unseen_row["path"]).convert("RGB").resize((SIZE_HIGH, SIZE_HIGH)), dtype=np.float32) / 255.0
malignant_probability = float(sum(unseen_probs[CLASS_TO_INDEX[label]] for label in ("MEL", "BCC", "AKIEC")))
fig, (image_axis, bar_axis) = plt.subplots(1, 2, figsize=(12, 4.5), gridspec_kw={"width_ratios": [1, 1.4]})
image_axis.imshow(unseen_image)
image_axis.set_title(f"unseen image {unseen_row['image']}\ntrue class: {unseen_row['label']}", fontsize=9)
image_axis.axis("off")
bar_axis.barh(CLASS_NAMES[::-1], unseen_probs[::-1], color="tab:blue")
for index, probability in enumerate(unseen_probs[::-1]):
    bar_axis.text(probability + 0.01, index, f"{probability:.3f}", va="center", fontsize=8)
bar_axis.set_xlim(0, 1.05)
bar_axis.set_xlabel("softmax probability")
bar_axis.set_title(f"predicted {CLASS_NAMES[int(unseen_probs.argmax())]} | "
                   f"P(malignant) = {malignant_probability:.3f}", fontsize=10)
fig.suptitle("2.19  probabilities for one unseen ISIC 2019 image", fontsize=11)
fig.tight_layout()
fig.savefig(FIGURES / "unseen_image_probabilities.png", dpi=150, bbox_inches="tight")
plt.show()
print("unseen image:", unseen_row["image"], "| true:", unseen_row["label"],
      "| predicted:", CLASS_NAMES[int(unseen_probs.argmax())], "| P(malignant):", round(malignant_probability, 3))


# 2.20 - Temperature scaling fitted on the validation split, then reliability diagram and ECE on test.
def expected_calibration_error(probabilities, labels, bins=10):
    confidence = probabilities.max(axis=1)
    correct = (probabilities.argmax(axis=1) == labels).astype(float)
    edges = np.linspace(0.0, 1.0, bins + 1)
    rows = []
    for lower, upper in zip(edges[:-1], edges[1:]):
        mask = (confidence > lower) & (confidence <= upper)
        if mask.sum() == 0:
            continue
        rows.append({"bin_center": (lower + upper) / 2, "count": int(mask.sum()),
                     "mean_confidence": float(confidence[mask].mean()),
                     "accuracy": float(correct[mask].mean())})
    table = pd.DataFrame(rows)
    ece = float(np.sum(table["count"] / len(labels) * (table["accuracy"] - table["mean_confidence"]).abs()))
    return table, ece


def apply_temperature(probabilities, temperature):
    logits = np.log(np.clip(probabilities, 1e-12, 1.0)) / temperature
    logits -= logits.max(axis=1, keepdims=True)
    scaled = np.exp(logits)
    return scaled / scaled.sum(axis=1, keepdims=True)


def negative_log_likelihood(probabilities, labels):
    return float(-np.mean(np.log(np.clip(probabilities[np.arange(len(labels)), labels], 1e-12, 1.0))))


val_probs, val_labels = ensemble_probabilities(val_ds_high)
temperatures = np.linspace(0.5, 5.0, 91)
losses = [negative_log_likelihood(apply_temperature(val_probs, t), val_labels) for t in temperatures]
best_temperature = float(temperatures[int(np.argmin(losses))])
print(f"temperature fitted on the validation split: {best_temperature:.2f} | "
      f"validation NLL {min(losses):.4f} (uncalibrated {negative_log_likelihood(val_probs, val_labels):.4f})")

reliability_uncalibrated, ece_uncalibrated = expected_calibration_error(test_probs, test_labels)
reliability_calibrated, ece_calibrated = expected_calibration_error(
    apply_temperature(test_probs, best_temperature), test_labels)
reliability_calibrated.to_csv(TABLES / "calibration_reliability.csv", index=False)
assert 0.0 <= ece_calibrated <= 1.0

fig, axis = plt.subplots(figsize=(6.5, 6))
axis.plot([0, 1], [0, 1], linestyle="--", color="grey", label="perfect calibration")
axis.plot(reliability_uncalibrated["mean_confidence"], reliability_uncalibrated["accuracy"], "o-",
          color="tab:red", label=f"uncalibrated (ECE {ece_uncalibrated:.4f})")
axis.plot(reliability_calibrated["mean_confidence"], reliability_calibrated["accuracy"], "o-",
          color="tab:blue", label=f"temperature scaled (ECE {ece_calibrated:.4f})")
axis.set_xlim(0, 1)
axis.set_ylim(0, 1)
axis.set_xlabel("mean predicted confidence")
axis.set_ylabel("empirical accuracy")
axis.set_title("2.20  reliability diagram (internal test, ensemble)")
axis.legend(loc="upper left", fontsize=8)
axis.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(FIGURES / "reliability_diagram.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"expected calibration error: uncalibrated {ece_uncalibrated:.4f} | calibrated {ece_calibrated:.4f}")
print("mean confidence on wrong predictions: %.4f | mean confidence on correct predictions: %.4f" % (
    test_confidence[test_labels != test_pred].mean(), test_confidence[test_labels == test_pred].mean()))

np.savez_compressed(TABLES / "external_predictions.npz", probs=external_probs, labels=external_labels)
np.savez_compressed(TABLES / "validation_predictions.npz", probs=val_probs, labels=val_labels)
external_metrics = {"images": int(len(external_labels)), "loss": external_loss, "accuracy": external_accuracy,
                    "balanced_accuracy": external_balanced_accuracy, "macro_f1": external_macro_f1,
                    "mean_confidence": float(external_confidence.mean()),
                    "temperature": best_temperature,
                    "calibration_ece_internal_test": ece_calibrated,
                    "calibration_ece_internal_test_uncalibrated": ece_uncalibrated}
(TABLES / "external_isic2019_metrics.json").write_text(json.dumps(external_metrics, indent=2))


## Artifacts, reproducibility and submission packaging

Required: one self-contained notebook and a verified results archive. Produces: weights, metrics JSON, all figures, results ZIP.

**Artifacts.** Everything the report needs is written with fixed names under `outputs/`:
`figures/` (augmentation, feature maps, gradient magnitudes, per-seed training curves, confusion
matrix, confident errors, unseen-image probabilities, reliability diagram), `tables/` (environment,
split manifest, data provenance, experiment configuration, model summary, three training histories,
curve analysis, classification report, test predictions, metrics, calibration table) and `model/`
(best, SWA and selected weights for each seed). `outputs/Group_8_Results.zip` bundles figures, tables
and the three selected weight files.


In [ ]:
# Artifacts, reproducibility and submission packaging.
import zipfile

metrics = {
    "experiment_config": json.loads((TABLES / "experiment_config.json").read_text()),
    "environment": json.loads((TABLES / "env.json").read_text()),
    "splits": {"train": int(len(train_df)), "validation": int(len(val_df)),
               "test": int(len(test_df)), "external_val": int(len(external_df))},
    "internal_test": json.loads((TABLES / "test_metrics.json").read_text()),
    "external_isic2019": json.loads((TABLES / "external_isic2019_metrics.json").read_text()),
    "selections": [json.loads((MODEL_ROOT / f"seed{seed}" / "selection.json").read_text()) for seed in SEEDS],
    "curve_analysis": (TABLES / "curve_analysis.txt").read_text(),
}
assert metrics["splits"]["train"] == expected_total
(TABLES / "metrics.json").write_text(json.dumps(metrics, indent=2))

archive_path = OUT_DIR / "Group_8_Results.zip"
if archive_path.is_file():
    archive_path.unlink()
with zipfile.ZipFile(archive_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for folder in (FIGURES, TABLES):
        for file_path in sorted(folder.rglob("*")):
            if file_path.is_file():
                archive.write(file_path, file_path.relative_to(OUT_DIR))
    for seed in SEEDS:
        archive.write(MODEL_ROOT / f"seed{seed}" / "selected.weights.h5", f"model/seed{seed}_selected.weights.h5")
with zipfile.ZipFile(archive_path) as archive:
    names = sorted(archive.namelist())
required = {"tables/experiment_config.json", "tables/data_provenance.json", "tables/metrics.json",
            "tables/train_history_seed8.csv", "tables/curve_analysis.txt", "tables/test_predictions.csv",
            "tables/calibration_reliability.csv", "figures/training_curves.png",
            "figures/confusion_matrix.png", "figures/reliability_diagram.png",
            "figures/unseen_image_probabilities.png", "figures/feature_maps.png",
            "figures/gradient_magnitudes_vgg_vs_resnet.png", "figures/augmentation_examples.png",
            "figures/top_confident_errors.png", "figures/validation_metrics.png"}
for seed in SEEDS:
    required.add(f"model/seed{seed}_selected.weights.h5")
missing = sorted(required - set(names))
assert not missing, f"missing artifacts in the zip: {missing}"
print(f"results archive: {archive_path} ({archive_path.stat().st_size / 1024 / 1024:.1f} MB, {len(names)} files)")
print("\n".join(names))
